# CPython Dictionary Internals, Hash Tables, and Interview Guide

## 1. THEORY LAYER
### Origin and Motivation
The dictionary is Python's core namespaces driver. Classes, modules, variables, and stack frames are mapped inside dictionaries.

### Memory Model
CPython uses a compact dictionary implementation (PEP 468) consisting of two arrays:
- Indices array: Contains offset offsets of keys in the entries array.
- Entries array: Contains `(hash, key, value)` pairs in insertion order.

---

## 2. CPYTHON INTERNAL LAYER
### C struct Definition (`dictobject.h`)
```c
typedef struct {
    PyObject_HEAD
    Py_ssize_t ma_used;          // Number of items
    uint64_t ma_version_tag;     // Global version tag
    PyDictKeysObject *ma_keys;   // Keys array (contains indices + entries)
    PyObject **ma_values;        // Non-null for split table dicts
} PyDictObject;
```

### Compact Dict Layout
```
indices = [ -1, 0, -1, 1, -1, -1, 2 ]
entries = [
    {hash1, key1, value1},
    {hash2, key2, value2},
    {hash3, key3, value3}
]
```

---


In [ ]:
import sys

print("=" * 70)
print("CPython Dictionary Resizing Simulation")
print("=" * 70)

d = {}
prev_size = sys.getsizeof(d)
for i in range(40):
    d[i] = i
    curr_size = sys.getsizeof(d)
    if curr_size != prev_size:
        print(f"Items: {len(d):>2} | Memory: {curr_size:>3} bytes | Resize Triggered!")
        prev_size = curr_size



---
## 3. COMPLETE METHOD REFERENCE

### 3.1 `get(key[, default])`
- **Syntax**: `dict.get(key)`
- **Time Complexity**: $O(1)$ average.


In [ ]:
print("\n=== Method: get ===")
d = {"a": 1}
print(f"get('a') -> {d.get('a')}")
print(f"get('b', 404) -> {d.get('b', 404)}")



### 3.2 `setdefault(key[, default])`
- **Syntax**: `dict.setdefault(key, default)`
- **Behavior**: Returns value if key is present, otherwise inserts key with default and returns it.


In [ ]:
print("\n=== Method: setdefault ===")
d = {"a": 1}
val = d.setdefault("b", 2)
print(f"setdefault('b', 2) returned: {val} | dict: {d}")



### 3.3 `update([other])`
- **Syntax**: `dict.update(other)`
- **Time Complexity**: $O(k)$ where $k$ is size of input.


In [ ]:
print("\n=== Method: update ===")
d = {"a": 1}
d.update({"b": 2, "c": 3})
print(f"update -> {d}")



### 3.4 `pop(key[, default])`
- **Syntax**: `dict.pop(key)`
- **Time Complexity**: $O(1)$.


In [ ]:
print("\n=== Method: pop ===")
d = {"a": 1, "b": 2}
print(f"pop('a') -> {d.pop('a')}, dict: {d}")



### 3.5 `popitem()`
- **Syntax**: `dict.popitem()`
- **Behavior**: Removes and returns last inserted key-value pair.


In [ ]:
print("\n=== Method: popitem ===")
d = {"a": 1, "b": 2}
print(f"popitem() -> {d.popitem()}, dict: {d}")



---
## 4. IMPLEMENTATION LAYER
### Level 1: Pure Python Reimplementation of a Compact Dictionary


In [ ]:
class CustomCompactDict:
    """Manual implementation of a Compact Hash Map in Pure Python."""
    def __init__(self, capacity=8):
        self.capacity = capacity
        # Indices table holds indices into the entries array
        self.indices = [-1] * self.capacity
        self.entries = []
        self.count = 0

    def _hash(self, key):
        return hash(key) % self.capacity

    def put(self, key, value):
        # Resize check
        if self.count / self.capacity > 0.66:
            self._resize()

        h_val = hash(key)
        idx = h_val % self.capacity
        
        # Check collision resolution
        while self.indices[idx] != -1:
            entry_idx = self.indices[idx]
            if self.entries[entry_idx][1] == key:
                # Update existing key value
                self.entries[entry_idx] = (h_val, key, value)
                return
            idx = (idx + 1) % self.capacity

        # Insert new key
        self.indices[idx] = len(self.entries)
        self.entries.append((h_val, key, value))
        self.count += 1

    def get(self, key, default=None):
        idx = self._hash(key)
        start_idx = idx
        while self.indices[idx] != -1:
            entry_idx = self.indices[idx]
            if self.entries[entry_idx][1] == key:
                return self.entries[entry_idx][2]
            idx = (idx + 1) % self.capacity
            if idx == start_idx:
                break
        return default

    def _resize(self):
        old_entries = self.entries
        self.capacity *= 2
        self.indices = [-1] * self.capacity
        self.entries = []
        self.count = 0
        for h, k, v in old_entries:
            self.put(k, v)

    def __repr__(self):
        items_str = ", ".join(f"{repr(k)}: {repr(v)}" for h, k, v in self.entries)
        return "{" + items_str + "}"

print("\n=== Level 1: Custom Compact Dict ===")
c_dict = CustomCompactDict()
c_dict.put("host", "127.0.0.1")
c_dict.put("port", 8080)
print(f"Dict: {c_dict} | Get port: {c_dict.get('port')}")



### Level 2: Python Built-in Comparison
Verify built-in dictionary behavior.


In [ ]:
print("\n=== Level 2: Dict views and iteration ===")
d = {"a": 1, "b": 2}
print(f"Keys view:   {d.keys()}")
print(f"Values view: {d.values()}")



### Level 3: Advanced Usage Systems — LRU Cache via Dict


In [ ]:
class SimpleLRUCache:
    """Least Recently Used (LRU) Cache utilizing dictionary ordering properties."""
    def __init__(self, maxsize=3):
        self.maxsize = maxsize
        self.cache = {}

    def get(self, key):
        if key not in self.cache:
            return None
        # Move accessed item to end of dict (recent)
        val = self.cache.pop(key)
        self.cache[key] = val
        return val

    def put(self, key, value):
        if key in self.cache:
            self.cache.pop(key)
        elif len(self.cache) >= self.maxsize:
            # Pop first key (oldest inserted)
            oldest = next(iter(self.cache))
            self.cache.pop(oldest)
            print(f"Evicted key: {oldest}")
        self.cache[key] = value

print("\n=== Level 3: LRU Cache ===")
cache = SimpleLRUCache()
cache.put("A", 1)
cache.put("B", 2)
cache.put("C", 3)
cache.put("D", 4) # Evicts A
print(f"Cache keys: {list(cache.cache.keys())}")



---
## 5. EXPERIMENTATION LAYER
### Hash collision slowdown test


In [ ]:
print("\n=== Section 5: Hash Collision Performance ===")
import time

class CollideKey:
    def __init__(self, val):
        self.val = val
    def __hash__(self):
        return 99 # Force all to conflict
    def __eq__(self, other):
        return isinstance(other, CollideKey) and self.val == other.val

c_dict = {}
t0 = time.perf_counter()
for i in range(1000):
    c_dict[CollideKey(i)] = i
t1 = time.perf_counter()
print(f"Time to insert 1000 colliding keys: {(t1 - t0)*1000:.2f} ms")



---
## 6. VISUALIZATION LAYER
```
Compact Dictionary Memory Split:
dk_indices: [ 0 | -1 | 2 | 1 | -1 ]
dk_entries: [
   0: (hashA, keyA, valA),
   1: (hashB, keyB, valB),
   2: (hashC, keyC, valC)
]
```
---
## 7. INTERVIEW MASTER LAYER

### 50 Scenario-Based Interview Q&As (Summary Selection)

1. **Why does Python dict iterate in insertion order?**
   - Because iteration scans the compact entries array sequentially, which appends elements in chronological order.
2. **What is a collision dummy/tombstone?**
   - When deleting a key from a hash table, its index cannot be simply set to -1 (empty) because it might break linear probing chains. It is marked as -2 (dummy/tombstone) to keep chains searchable.
3. **Explain compact dictionary memory advantages.**
   - By using small bytes arrays for indexing indices instead of sparse pointer arrays, memory overhead drops by ~30%.
